### TSC GRPO 训练 - new_prompt_training JSON 协议（保留仿真回溯评估）



In [1]:
import os
os.environ["UNSLOTH_USE_MODELSCOPE"] = "1"



In [ ]:
import subprocess
result = subprocess.run('bash -c "source /etc/network_turbo && env | grep proxy"', shell=True, capture_output=True, text=True)
for line in result.stdout.splitlines():
    if '=' in line:
        var, value = line.split('=', 1)
        os.environ[var] = value



### 加载模型



In [2]:
from unsloth import FastLanguageModel
import torch
import os

max_seq_length = 2048  # 增加序列长度以容纳完整的 prompt + completion
lora_rank = 32

os.environ["HF_HOME"] = 'model'
os.environ["MODELSCOPE_CACHE"] = 'model'

# ====== 断点续训：有 checkpoint 就继续微调；没有就从 base 模型开始 ======
BASE_MODEL_DIR = "model/models/qwen3-4B-SFT"
CHECKPOINT_DIR = "checkpoints/newprompt_grpo_latest"

def _looks_like_checkpoint(path: str) -> bool:
    if not os.path.isdir(path):
        return False
    # PEFT/LoRA 常见标志文件（任意一个即可认为可加载）
    marker_files = [
        "adapter_config.json",
        "adapter_model.safetensors",
        "adapter_model.bin",
        "config.json",
    ]
    return any(os.path.isfile(os.path.join(path, f)) for f in marker_files)

resume_from = CHECKPOINT_DIR if _looks_like_checkpoint(CHECKPOINT_DIR) else BASE_MODEL_DIR
if resume_from == CHECKPOINT_DIR:
    print(f"✓ 检测到 checkpoint，将从此继续微调: {CHECKPOINT_DIR}")
else:
    print(f"ℹ 未检测到 checkpoint，将从基础模型开始: {BASE_MODEL_DIR}")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = resume_from,
    max_seq_length = max_seq_length,
    load_in_4bit = False,
    fast_inference = False,
    max_lora_rank = lora_rank,
    gpu_memory_utilization = 0.8,
)

# 若是从 base 模型开始，需要创建 LoRA；若是从 checkpoint 加载，一般已包含 LoRA（无需重复包一层）
if resume_from == BASE_MODEL_DIR:
    model = FastLanguageModel.get_peft_model(
        model,
        r = lora_rank,
        target_modules = [
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
        lora_alpha = lora_rank * 2,
        use_gradient_checkpointing = "unsloth",
        random_state = 3407,
    )
else:
    # 尽量开启梯度检查点（如果当前模型类型支持）
    try:
        model.gradient_checkpointing_enable()
    except Exception:
        pass

    # 保险：有些版本从 checkpoint 加载后 LoRA 参数可能默认不可训练
    _trainable = [p for p in model.parameters() if p.requires_grad]
    if len(_trainable) == 0:
        for name, p in model.named_parameters():
            if "lora" in name.lower():
                p.requires_grad = True
        print("⚠️ checkpoint 加载后未检测到可训练参数，已强制启用 LoRA 参数训练")



🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
ℹ 未检测到 checkpoint，将从基础模型开始: model/models/qwen3-4B-SFT
==((====))==  Unsloth 2025.12.9: Fast Qwen3 patching. Transformers: 4.57.3.
   \\   /|    NVIDIA GB10. Num GPUs = 1. Max memory: 119.698 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0.dev20251228+cu130. CUDA: 12.1. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Unsloth 2025.12.9 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


### 场景发现 & 仿真器



In [3]:
import json
import random
import sys
import xml.etree.ElementTree as ET
from collections import deque
from typing import Dict, List, Tuple


def discover_environments(environments_root: str) -> Dict[str, Dict]:
    environments: Dict[str, Dict] = {}
    if not os.path.isdir(environments_root):
        print(f"警告: environments 目录不存在: {environments_root}")
        return environments

    for scenario_name in sorted(os.listdir(environments_root)):
        scenario_dir = os.path.join(environments_root, scenario_name)
        if not os.path.isdir(scenario_dir) or scenario_name.startswith('.'):
            continue

        sumocfg = None
        for f in os.listdir(scenario_dir):
            if f.endswith('.sumocfg'):
                sumocfg = os.path.join(scenario_dir, f)
                break

        net_xml = None
        for f in os.listdir(scenario_dir):
            if f.endswith('.net.xml'):
                net_xml = os.path.join(scenario_dir, f)
                break

        if sumocfg and net_xml:
            tl_ids = extract_traffic_light_ids(net_xml)
            if tl_ids:
                environments[scenario_name] = {
                    'sumocfg': sumocfg,
                    'net': net_xml,
                    'tl_ids': tl_ids,
                }

    return environments


def extract_traffic_light_ids(net_xml_path: str) -> List[str]:
    tl_ids: List[str] = []
    try:
        for _event, elem in ET.iterparse(net_xml_path, events=("end",)):
            if elem.tag == "tlLogic":
                tl_id = elem.attrib.get("id")
                if tl_id and tl_id not in tl_ids:
                    tl_ids.append(tl_id)
                elem.clear()
    except Exception as e:
        print(f"解析 {net_xml_path} 失败: {e}")
    return tl_ids


ENVIRONMENTS_ROOT = os.path.join(os.getcwd(), 'sumo_simulation/environments')
AVAILABLE_ENVIRONMENTS = discover_environments(ENVIRONMENTS_ROOT)
print(f"发现场景数: {len(AVAILABLE_ENVIRONMENTS)}")
total_intersections = sum(len(env['tl_ids']) for env in AVAILABLE_ENVIRONMENTS.values())
print(f"总交叉口数量: {total_intersections}")

# 添加 sumo_simulation 目录到 Python 路径
sumo_sim_path = os.path.join(os.getcwd(), 'sumo_simulation')
if sumo_sim_path not in sys.path:
    sys.path.insert(0, sumo_sim_path)

from sumo_simulator import SUMOSimulator



发现场景数: 3
总交叉口数量: 75


### new_prompt 构造、解析、奖励（含回溯评估）



In [4]:
import datetime
import gc

from scu_tsc_newprompt.phase_parser import get_net_phase_minmax_one_based, get_phase_order_one_based
from scu_tsc_newprompt.constraint_sampler import sample_phase_limits_hybrid
from scu_tsc_newprompt.prompt_builder import build_cycle_predict_input_json, wrap_prompt_with_markers
from scu_tsc_newprompt.rewards import score_constraints_and_format


SYSTEM_PROMPT = """你是交通信号配时优化专家。
你将收到一个 JSON（用【cycle_predict_input_json】...【/cycle_predict_input_json】包裹）。
你的任务是：在满足硬约束前提下，输出下一周期各相位最终绿灯时间 final（单位：秒）。
注意：history 数据可能不完整（例如 only recent_cycles；yesterday_same_time/last_week_same_time 可能为 null），必须基于可用部分输出结果。
只输出最终 JSON 数组(list)，不要输出任何解释/过程。
"""

USER_INSTRUCTIONS = """任务（必须完成）：
1) 基于输入 JSON 的 history.* 历史数据，自行决定预测算法/模型/参数，预测“下一周期各相位需求强度”（仅在内部使用，不输出过程/中间值）。
2) 在满足硬约束前提下，输出下一周期各相位最终绿灯时间 final（单位：秒）。

要求（必须遵守）：
1) history 数据可能不完整（仅 recent_cycles；windows.yesterday_same_time / last_week_same_time 可能为 null）；这是正常情况，请基于可用部分完成预测，并且必须继续输出结果。
2) 若 history 中存在有效数据，请使用它完成预测并体现在结果中；不要忽略 history 数据随意分配。
3) 若 history 数据缺失/异常/几乎全为 0，仍需输出满足硬约束的可执行方案，但不得编造不存在的数据。
4) 只输出最终 JSON，不要输出任何解释/过程。

输出要求（必须严格遵守）：
1) 只输出最终 JSON（不要任何说明、不要 Markdown）。
2) JSON 顶层必须是数组(list)；数组长度必须等于相位数。
3) 数组元素必须为对象：{"phase_id": <int>, "final": <int>}；不允许输出其它字段。
4) phase_id 必须覆盖全部相位且不重复，并且顺序必须与 phase_order 完全一致。
"""


def build_user_prompt(payload: dict) -> str:
    return wrap_prompt_with_markers(payload) + "\n\n" + USER_INSTRUCTIONS


def collect_phase_waits_snapshot(simulator: SUMOSimulator, tl_id: str, phase_order: List[int]) -> List[dict]:
    """用当前时刻各相位 incoming lanes 的停止车辆数，作为 avg_wait 的代理。"""
    import traci

    waits = []
    for phase_id in phase_order:
        phase_idx = phase_id - 1
        lanes = simulator.get_phase_controlled_lanes(tl_id, phase_idx).get('incoming_lanes', [])
        if not lanes:
            avg = 0.0
        else:
            total = 0.0
            for ln in lanes:
                try:
                    total += traci.lane.getLastStepHaltingNumber(ln)
                except Exception:
                    pass
            avg = total / max(1, len(lanes))
        waits.append({'phase_id': int(phase_id), 'avg_wait': float(round(avg, 2))})
    return waits


def evaluate_plan_once(simulator: SUMOSimulator, tl_id: str, plan: List[dict]) -> dict:
    """执行一个整周期方案并返回指标（在当前仿真状态下直接前进）。"""
    import traci

    # 统一收集所有相位可能涉及的 incoming lanes（用于粗略 passed/queue 统计）
    phase_info = simulator.get_phase_info(tl_id)
    n = int(phase_info.get('num_phases', 0))
    all_lanes = set()
    for idx in range(n):
        all_lanes.update(simulator.get_phase_controlled_lanes(tl_id, idx).get('incoming_lanes', []))
    all_lanes = list(all_lanes)

    vehicles_before = set()
    for ln in all_lanes:
        try:
            vehicles_before.update(traci.lane.getLastStepVehicleIDs(ln))
        except Exception:
            pass

    total_queue_proxy = 0.0
    for step in plan:
        pid = int(step['phase_id'])
        dur = int(step['final'])
        traci.trafficlight.setPhase(tl_id, pid - 1)
        for _ in range(max(0, dur)):
            traci.simulationStep()
            q = 0.0
            for ln in all_lanes:
                try:
                    q += traci.lane.getLastStepHaltingNumber(ln)
                except Exception:
                    pass
            total_queue_proxy += q

    vehicles_after = set()
    queue_end = 0.0
    for ln in all_lanes:
        try:
            vehicles_after.update(traci.lane.getLastStepVehicleIDs(ln))
            queue_end += traci.lane.getLastStepHaltingNumber(ln)
        except Exception:
            pass

    passed = len(vehicles_before - vehicles_after)
    return {
        'passed_vehicles': float(passed),
        'queue_vehicles': float(queue_end),
        'total_queue_proxy': float(total_queue_proxy),
        'sim_time': float(traci.simulation.getTime()),
    }


def evaluate_multiple_plans_with_rollback(simulator: SUMOSimulator, tl_id: str, plans: List[List[dict]]) -> List[dict]:
    """对多个候选方案做回溯评估：同一检查点下逐个执行/恢复，最后回到原状态。"""
    state_file = simulator.save_simulation_state(tl_id)
    results = []
    try:
        for p in plans:
            simulator.restore_simulation_state(state_file)
            results.append(evaluate_plan_once(simulator, tl_id, p))
        simulator.restore_simulation_state(state_file)
    finally:
        simulator.cleanup_state_file(state_file)
    return results



### GRPO 训练（每步：构造 JSON prompt → 生成 N 个方案 → 回溯评估 → 更新 → 执行最优推进）



In [5]:
import torch.nn.functional as F
from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup

TRAIN_CONFIG = {
    'gui': False,
    'max_tl_per_scenario': 30,
    'warmup_steps': 80,
    'steps_per_tl': 60,
    'num_generations': 4,
    'learning_rate': 1e-6,
    'max_new_tokens': 256,
    'temperature': 0.8,
    'top_p': 0.95,
    'top_k': 50,
    'gradient_accumulation_steps': 4,
    'log_interval': 10,
    'clear_cache_interval': 5,
    # 仿真 reward 权重
    'alpha': 1.0,
    'beta': 0.5,
    # recent_cycles 窗口
    'recent_cycles_maxlen': 12,
}


def compute_grpo_loss(model, tokenizer, prompt_ids, completion_ids, rewards):
    device = next(model.parameters()).device
    
    # 截断 prompt 和 completion 以适应 max_seq_length
    max_len = max_seq_length
    prompt_len = prompt_ids.shape[1]
    completion_len = completion_ids.shape[1]
    total_len = prompt_len + completion_len
    
    if total_len > max_len:
        # 优先保留 completion，截断 prompt
        available_for_prompt = max(256, max_len - completion_len)  # 至少保留 256 个 prompt token
        if prompt_len > available_for_prompt:
            prompt_ids = prompt_ids[:, -available_for_prompt:]  # 保留 prompt 末尾
            prompt_len = available_for_prompt
        # 如果还是超长，截断 completion
        if prompt_len + completion_len > max_len:
            completion_ids = completion_ids[:, :max_len - prompt_len]
            completion_len = completion_ids.shape[1]
    
    input_ids = torch.cat([prompt_ids, completion_ids], dim=1).to(device)
    outputs = model(input_ids=input_ids)
    logits = outputs.logits

    # 确保维度匹配
    completion_logits = logits[:, prompt_len - 1 : prompt_len - 1 + completion_len, :]
    completion_targets = completion_ids.to(device)
    
    # 确保维度一致
    min_len = min(completion_logits.shape[1], completion_targets.shape[1])
    completion_logits = completion_logits[:, :min_len, :]
    completion_targets = completion_targets[:, :min_len]

    log_probs = F.log_softmax(completion_logits, dim=-1)
    token_log_probs = torch.gather(log_probs, 2, completion_targets.unsqueeze(-1)).squeeze(-1)

    attention_mask = (completion_targets != tokenizer.pad_token_id).float()
    seq_log_probs = (token_log_probs * attention_mask).sum(dim=1) / (attention_mask.sum(dim=1) + 1e-8)

    rewards_tensor = torch.tensor(rewards, dtype=torch.float32, device=device)
    rewards_normalized = (rewards_tensor - rewards_tensor.mean()) / (rewards_tensor.std() + 1e-8)

    policy_loss = -(seq_log_probs * rewards_normalized).mean()

    return policy_loss, {
        'policy_loss': float(policy_loss.item()),
        'mean_reward': float(rewards_tensor.mean().item()),
        'reward_std': float(rewards_tensor.std().item()),
    }


def build_scenario_tl_list() -> List[Tuple[str, str]]:
    pairs: List[Tuple[str, str]] = []
    for scenario_name, info in AVAILABLE_ENVIRONMENTS.items():
        for tl_id in info['tl_ids'][: TRAIN_CONFIG['max_tl_per_scenario']]:
            pairs.append((scenario_name, tl_id))
    random.shuffle(pairs)
    return pairs


def train_one_tl(scenario_name: str, tl_id: str) -> bool:
    env_info = AVAILABLE_ENVIRONMENTS[scenario_name]

    simulator = SUMOSimulator(
        config_file=env_info['sumocfg'],
        junctions_file=None,
        gui=TRAIN_CONFIG['gui'],
    )
    if not simulator.start_simulation():
        print(f"✗ 启动失败: {scenario_name}/{tl_id}")
        return False

    # warmup
    for _ in range(TRAIN_CONFIG['warmup_steps']):
        if simulator.is_connected():
            simulator.step()

    # phase order from net.xml (1-based)
    phase_order = get_phase_order_one_based(env_info['net'], tl_id)
    if not phase_order:
        print(f"✗ 未解析到相位: {scenario_name}/{tl_id}")
        simulator.close()
        return False

    net_minmax = get_net_phase_minmax_one_based(env_info['net'], tl_id)

    # history buffer: recent_cycles only
    recent_cycles_buf = deque(maxlen=TRAIN_CONFIG['recent_cycles_maxlen'])

    # optimizer
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = AdamW(trainable_params, lr=TRAIN_CONFIG['learning_rate'], weight_decay=0.1)
    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(TRAIN_CONFIG['steps_per_tl'] * 0.1),
        num_training_steps=TRAIN_CONFIG['steps_per_tl'],
    )
    optimizer.zero_grad()

    for step_idx in range(TRAIN_CONFIG['steps_per_tl']):
        if not simulator.is_connected():
            break

        # 随机化约束（每步变化），同一 (scenario, tl, step) 可复现
        rng = random.Random((hash(scenario_name) ^ hash(tl_id) ^ step_idx) & 0xFFFFFFFF)
        phase_limits = sample_phase_limits_hybrid(phase_order, net_minmax, rng=rng)

        payload = build_cycle_predict_input_json(
            scenario_name=scenario_name,
            tl_id=tl_id,
            phase_order=phase_order,
            phase_limits=phase_limits,
            recent_cycles=list(recent_cycles_buf),
            include_windows_recent_past=True,
            windows_recent_past=None,
        )

        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": build_user_prompt(payload)},
        ]
        prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        prompt_ids = tokenizer(prompt_text, return_tensors="pt", add_special_tokens=False).input_ids

        device = next(model.parameters()).device
        prompt_ids_batch = prompt_ids.repeat(TRAIN_CONFIG['num_generations'], 1).to(device)

        # generate
        FastLanguageModel.for_inference(model)
        with torch.no_grad():
            generated = model.generate(
                input_ids=prompt_ids_batch,
                max_new_tokens=TRAIN_CONFIG['max_new_tokens'],
                temperature=TRAIN_CONFIG['temperature'],
                top_p=TRAIN_CONFIG['top_p'],
                top_k=TRAIN_CONFIG['top_k'],
                do_sample=True,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
        completion_ids = generated[:, prompt_ids.shape[1] :].clone().detach()
        FastLanguageModel.for_training(model)
        completions = tokenizer.batch_decode(completion_ids, skip_special_tokens=True)

        # constraint score + parse（软约束检查，只检查 min_green/max_green）
        parsed_plans: List[List[dict] | None] = []
        constraint_scores: List[float] = []
        for i, txt in enumerate(completions):
            s, _info, plan = score_constraints_and_format(
                completion_text=txt,
                phase_order=phase_order,
                phase_limits=phase_limits,
            )
            constraint_scores.append(float(s))
            parsed_plans.append(plan)
            
            # === 调试：每10 steps 打印所有 completion ===
            if step_idx % 10 == 0:
                print(f"\n[DEBUG step={step_idx} gen={i}] score={s:.2f} error={_info.get('error')}")
                print(f"  bounds_penalty={_info.get('bounds_penalty', 0):.3f}")
                print(f"  completion={repr(txt[:300])}")

        # 回溯仿真评估：只对“满足全部硬约束”的候选做回溯（省算力）
        plans_for_sim: List[List[dict]] = []
        sim_map: Dict[int, int] = {}
        for i, p in enumerate(parsed_plans):
            if p is not None:  # 只要 plan 解析成功就进行仿真评估（软约束）
                sim_map[i] = len(plans_for_sim)
                plans_for_sim.append(p)

        # 仿真评估：计算交通指标
        sim_rewards = [0.0] * len(completions)
        if plans_for_sim:
            sim_results = evaluate_multiple_plans_with_rollback(simulator, tl_id, plans_for_sim)
            for i in sim_map:
                r = sim_results[sim_map[i]]
                passed = float(r.get('passed_vehicles', 0.0))
                queue = float(r.get('queue_vehicles', 0.0))
                total_queue_proxy = float(r.get('total_queue_proxy', 0.0))
                # 仿真 reward：passed越多越好，queue越少越好，total_queue_proxy越少越好
                sim_reward = (TRAIN_CONFIG['alpha'] * passed - TRAIN_CONFIG['beta'] * queue - 0.01 * total_queue_proxy) / 10.0
                sim_rewards[i] = float(sim_reward)

        # total reward = 约束分数 + 仿真 reward
        rewards = [constraint_scores[i] + sim_rewards[i] for i in range(len(completions))]

        # GRPO update
        prompt_ids_for_loss = prompt_ids_batch.clone()
        completion_ids_for_loss = completion_ids.clone()
        loss, info = compute_grpo_loss(model, tokenizer, prompt_ids_for_loss, completion_ids_for_loss, rewards)
        (loss / TRAIN_CONFIG['gradient_accumulation_steps']).backward()

        if (step_idx + 1) % TRAIN_CONFIG['gradient_accumulation_steps'] == 0:
            torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        # 执行最优（推进单条真实轨迹）
        best_i = max(range(len(rewards)), key=lambda i: rewards[i])
        best_plan = parsed_plans[best_i]

        if best_plan is None:
            for _ in range(10):
                if simulator.is_connected():
                    simulator.step()
        else:
            waits = collect_phase_waits_snapshot(simulator, tl_id, phase_order)
            recent_cycles_buf.append(
                {
                    'time': datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
                    'phase_waits': waits,
                }
            )
            _ = evaluate_plan_once(simulator, tl_id, best_plan)

        if (step_idx + 1) % TRAIN_CONFIG['log_interval'] == 0:
            valid_plans = sum(1 for p in parsed_plans if p is not None)
            print(
                f"[{scenario_name}/{tl_id}] step {step_idx+1}/{TRAIN_CONFIG['steps_per_tl']} "
                f"loss={info['policy_loss']:.4f} mean_r={info['mean_reward']:.3f} std={info['reward_std']:.3f} "
                f"valid={valid_plans}/{len(parsed_plans)} best_constraint={constraint_scores[best_i]:.2f} best_sim={sim_rewards[best_i]:.2f}"
            )

        # cleanup
        del loss, generated, completion_ids_for_loss, prompt_ids_for_loss
        gc.collect()
        if torch.cuda.is_available() and (step_idx + 1) % TRAIN_CONFIG['clear_cache_interval'] == 0:
            torch.cuda.empty_cache()

    simulator.close()
    return True


pairs = build_scenario_tl_list()
print(f"训练组合数: {len(pairs)}")

# 先跑前若干个路口（避免一次训练过大）
MAX_TRAIN_TLS = 50
import shutil
save_dir = "checkpoints/newprompt_grpo_latest"

for i, (sc, tl) in enumerate(pairs[:MAX_TRAIN_TLS], start=1):
    print(f"\n=== [{i}/{min(MAX_TRAIN_TLS, len(pairs))}] {sc}/{tl} ===")
    _ok = train_one_tl(sc, tl)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    # 每训练完一个交叉口保存一次 checkpoint（覆盖保存）
    if os.path.isdir(save_dir):
        shutil.rmtree(save_dir)
    os.makedirs(save_dir, exist_ok=True)
    model.save_pretrained(save_dir)
    print(f"✓ checkpoint 已保存到 {save_dir} (完成 {i}/{min(MAX_TRAIN_TLS, len(pairs))} 个交叉口)")



训练组合数: 59

=== [1/50] ingolstadt21/30624898 ===
DEBUG: 使用的 SUMO 路径是: /usr/share/sumo/bin/sumo
Starting SUMO with command: /usr/share/sumo/bin/sumo -c /home/samuel/unsloth_dgx/workspace/SCU_TSC/sumo_simulation/environments/ingolstadt21/ingolstadt21.sumocfg --step-length 1.0 --no-warnings true --start
 Retrying in 1 seconds
Successfully connected to SUMO
Starting warmup phase...
Warmup progress: 0/300
Warmup progress: 100/300
Warmup progress: 200/300
Warmup completed. Starting real-time simulation.


/home/samuel/unsloth_dgx/workspace/SCU_TSC/venv/lib/python3.13/site-packages/unsloth/kernels/utils.py:963: UserWarning: An output with one or more elements was resized since it had shape [1, 4, 2560], which does not match the required output shape [4, 1, 2560]. This behavior is deprecated, and in a future PyTorch release outputs will not be resized unless they have zero elements. You can explicitly reuse an out tensor t by resizing it, inplace, to zero elements with t.resize_(0). (Triggered internally at /pytorch/aten/src/ATen/native/Resize.cpp:31.)
  out = torch_matmul(X, W.t(), out = out)



[DEBUG step=0 gen=0] score=1.00 error=None
  bounds_penalty=0.000
  completion='[{"phase_id":1,"final":60},{"phase_id":2,"final":20},{"phase_id":3,"final":45},{"phase_id":4,"final":18},{"phase_id":5,"final":30},{"phase_id":6,"final":75}]'

[DEBUG step=0 gen=1] score=1.00 error=None
  bounds_penalty=0.000
  completion='[{"phase_id":1,"final":45},{"phase_id":2,"final":20},{"phase_id":3,"final":55},{"phase_id":4,"final":19},{"phase_id":5,"final":12},{"phase_id":6,"final":78}]'

[DEBUG step=0 gen=2] score=1.00 error=None
  bounds_penalty=0.000
  completion='[{"phase_id":1,"final":65},{"phase_id":2,"final":19},{"phase_id":3,"final":44},{"phase_id":4,"final":23},{"phase_id":5,"final":12},{"phase_id":6,"final":85}]'

[DEBUG step=0 gen=3] score=1.00 error=None
  bounds_penalty=0.000
  completion='[{"phase_id":1,"final":52},{"phase_id":2,"final":17},{"phase_id":3,"final":41},{"phase_id":4,"final":20},{"phase_id":5,"final":16},{"phase_id":6,"final":79}]'
Unsloth: Will smartly offload gradients 

FatalTraCIError: Connection closed by SUMO.